### Задание

1. Выбрать QA датасет (https://huggingface.co/datasets). 
2. На выбранном датасете обучить модель https://huggingface.co/intfloat/multilingual-e5-small с помощью этого кода.
3. Переписать код под актуальную версию sentence-transformers.
4. Сравнить результаты

In [1]:
# pip install transformers==4.45.2 
# pip install sentence-transformers==3.1.1

In [3]:
import pickle
from random import shuffle

from datasets import load_dataset
from sentence_transformers import InputExample

In [4]:
import torch

device = 'mps' if torch.backends.mps.is_built() else 'cuda' if torch.cuda.is_available() else 'cpu'

print(device)

mps


In [17]:
from sentence_transformers import SentenceTransformer, models, evaluation


def raw_bi_encoder():
    word_embedding_model = models.Transformer('cointegrated/rubert-tiny2', max_seq_length=256)
    pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension())

    bi_encoder = SentenceTransformer(
        modules=[word_embedding_model, pooling_model],
        device=device,
    )

    return bi_encoder

In [8]:
bi_encoder = raw_bi_encoder()

print(bi_encoder)

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 312, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)


## Служебные функции для работы с данными

In [12]:
def shrink_repeated_samples(
        queries,
        docs,
        labels,
):
    docs_was = set()
    qs = []
    ds = []
    ls = []
    for i in range(len(queries)):
        q, d, l = queries[i], docs[i], labels[i]
        if d in docs_was:
            continue
        qs.append(q)
        ds.append(d)
        ls.append(l)
        docs_was.add(d)

    return qs, ds, ls


def trainval_shuffled_data(from_file=True):
    datafile = 'trainval.pkl'

    if from_file:
        with open(datafile, 'rb') as f:
            data = pickle.load(f)
        return data

    good, bad = load_trainval_samples()
    data = good + bad
    shuffle(data)

    with open(datafile, 'wb') as f:
        pickle.dump(data, f)

    return data


def test_shuffled_data(from_file=True):
    datafile = 'test.pkl'

    if from_file:
        with open(datafile, 'rb') as f:
            data = pickle.load(f)
        return data

    good, bad = load_test_samples()
    data = good + bad

    with open(datafile, 'wb') as f:
        pickle.dump(data, f)

    return data


def trainval_examples():
    data = trainval_shuffled_data(from_file=True)

    trainval = []
    for question, document, cos in data:
        trainval.append(
            InputExample(texts=[question, document], label=cos)
        )

    return trainval


def load_test_samples():
    _, _, test = _load_sberquad_dataset()

    questions = test['question'].tolist()
    contexts = test['context'].tolist()

    return _inflate_with_negative_samples(
        questions=questions,
        contexts=contexts,
    )


def load_trainval_samples():
    train, val, _ = _load_sberquad_dataset()

    questions = train['question'].tolist() + val['question'].tolist()
    contexts = train['context'].tolist() + val['context'].tolist()

    return _inflate_with_negative_samples(
        questions=questions,
        contexts=contexts,
    )


def _inflate_with_negative_samples(
        questions,
        contexts,
        delta=30,  # in sberquad dataset same contexts are close
):
    good_q_c_cos = list(zip(questions, contexts, [1.0] * len(questions)))

    bad_q_c_cos = []
    n = len(good_q_c_cos)
    for i in range(n):
        cur_q, cur_c, _ = good_q_c_cos[i]
        next_q, next_c, _ = good_q_c_cos[(i + delta) % n]
        if next_c != cur_c:
            bad_q_c_cos.append((cur_q, next_c, 0.0))

    return good_q_c_cos, bad_q_c_cos


def _load_raw_sberquad_dataset(from_file=True):
    sberquad = 'sberquad.pkl'
    if from_file:
        with open(sberquad, 'rb') as f:
            df = pickle.load(f)
    else:
        df = load_dataset('sberquad')
        with open(sberquad, 'wb') as f:
            pickle.dump(df, f)

    return df


def _load_sberquad_dataset():
    df = _load_raw_sberquad_dataset()
    train = df['train'].to_pandas()
    validation = df['validation'].to_pandas()
    test = df['test'].to_pandas()

    return train, validation, test

# Метрики близости без дообучения на trainval

In [10]:
trainval = trainval_shuffled_data(from_file=True)

n_samples = 1000
# questions, document_contexts, coss = zip(*trainval[:n_samples])
questions, document_contexts, coss = zip(*trainval)

In [18]:
evaluator = evaluation.EmbeddingSimilarityEvaluator(questions, document_contexts, coss)

In [19]:
bi_encoder.evaluate(evaluator)

{'pearson_cosine': 0.7693105523306658,
 'spearman_cosine': 0.7884154865860669,
 'pearson_manhattan': 0.6255220209027033,
 'spearman_manhattan': 0.6456981482352,
 'pearson_euclidean': 0.6211237827157337,
 'spearman_euclidean': 0.6412993295276571,
 'pearson_dot': 0.7570034025575352,
 'spearman_dot': 0.793044605008729,
 'pearson_max': 0.7693105523306658,
 'spearman_max': 0.793044605008729}

# Метрики близости без дообучения на test

In [20]:
n_test_samples = 1000
test = test_shuffled_data(from_file=True)
test_questions, test_document_contexts, test_coss = zip(*test[:n_test_samples])

In [21]:
evaluator = evaluation.EmbeddingSimilarityEvaluator(test_questions, test_document_contexts, test_coss)

In [22]:
bi_encoder = raw_bi_encoder()

/Users/ivanguseff/VScode/conv_ai_llms/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [23]:
bi_encoder.evaluate(evaluator)

{'pearson_cosine': 0.7757868135164379,
 'spearman_cosine': 0.8022349576047765,
 'pearson_manhattan': 0.6253880558443918,
 'spearman_manhattan': 0.650982554692017,
 'pearson_euclidean': 0.6269351588053115,
 'spearman_euclidean': 0.6528116341830604,
 'pearson_dot': 0.7661326789941487,
 'spearman_dot': 0.8064889530877336,
 'pearson_max': 0.7757868135164379,
 'spearman_max': 0.8064889530877336}

# Дообучение

In [24]:
# trainval_examples = trainval_examples()[:n_samples]
trainval_examples = trainval_examples()
print(trainval_examples[0].__dict__)

{'guid': '', 'texts': ['что оказала Заметное, хотя и менее значительное влияние на воззрения Локка', 'Заметное, хотя и менее значительное влияние на воззрения Локка оказала психология Гоббса, у которого заимствован, например, порядок изложения Опыта . Описывая процессы сравнения, Локк следует за Гоббсом; вместе с ним он утверждает, что отношения не принадлежат вещам, а составляют результат сравнения, что отношений бесчисленное множество, что более важные отношения суть тождество и различие, равенство и неравенство, сходство и несходство, смежность по пространству и времени, причина и действие. В трактате о языке, то есть в третьей книге Опыта , Локк развивает мысли Гоббса. В учении о воле Локк находится в сильнейшей зависимости от Гоббса; вместе с последним он учит, что стремление к удовольствию есть единственное проходящее через всю нашу психическую жизнь и что понятие о добре и зле у различных людей совершенно различно. В учении о свободе воли Локк вместе с Гоббсом утверждает, что во

In [25]:
from torch.utils.data import DataLoader

trainval_dataloader_set = DataLoader(trainval_examples, shuffle=True, batch_size=32,
                                     collate_fn=bi_encoder.smart_batching_collate)

In [26]:
from sentence_transformers import losses

trainval_loss_set = losses.CosineSimilarityLoss(bi_encoder)

In [27]:
trainval_loss_set

CosineSimilarityLoss(
  (model): SentenceTransformer(
    (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
    (1): Pooling({'word_embedding_dimension': 312, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  )
  (loss_fct): MSELoss()
  (cos_score_transformation): Identity()
)

In [28]:
(query_batch, context_batch), labels = next(iter(trainval_dataloader_set))
query_batch['input_ids'].shape, context_batch['input_ids'].shape, labels.shape

(torch.Size([32, 43]), torch.Size([32, 256]), torch.Size([32]))

In [29]:
bi_encoder.fit(
    train_objectives=[(trainval_dataloader_set, trainval_loss_set)],
    output_path='qa/results',
    epochs=5,
    evaluator=evaluator,
)

  0%|          | 2/15740 [00:08<17:22:27,  3.97s/it]

KeyboardInterrupt: 

# Загрузка чекпоинта

In [ ]:
from sentence_transformers import SentenceTransformer

finetuned_bi_encoder = SentenceTransformer('qa/results')

In [ ]:
finetuned_bi_encoder.evaluate(evaluator)

{'pearson_cosine': 0.8793680890837718,
 'spearman_cosine': 0.844470065852506,
 'pearson_manhattan': 0.8772645432422359,
 'spearman_manhattan': 0.8462298620294948,
 'pearson_euclidean': 0.8773090888287658,
 'spearman_euclidean': 0.8456755955170574,
 'pearson_dot': 0.8593191423803976,
 'spearman_dot': 0.8383731342156948,
 'pearson_max': 0.8793680890837718,
 'spearman_max': 0.8462298620294948}

# Расчёт метрик

In [ ]:
data = test_shuffled_data(from_file=True)

In [ ]:
n_test_samples = 1000
queries, docs, labels = zip(*data[:n_test_samples])

In [ ]:
print('docs count before shrinking: ', len(docs))

docs count before shrinking:  1000


In [ ]:
docs = list(set(docs))

In [ ]:
print('docs count without duplicates: ', len(docs))

docs count without duplicates:  891


In [ ]:
doc_embs = finetuned_bi_encoder.encode(
    docs,
    convert_to_tensor=True,
    show_progress_bar=True,
)

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

In [ ]:
query_embs = finetuned_bi_encoder.encode(
    queries,
    convert_to_tensor=True,
    show_progress_bar=True,
)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [ ]:
from sentence_transformers import util

In [ ]:
cos_sim_score = util.semantic_search(query_embs, doc_embs, top_k=3)

In [ ]:
for_first_query = cos_sim_score[0]
print(for_first_query)

[{'corpus_id': 561, 'score': 0.8310010433197021}, {'corpus_id': 552, 'score': 0.7761925458908081}, {'corpus_id': 139, 'score': 0.7383377552032471}]


In [ ]:
print('Question: ', queries[0])
print()
for no, ir in enumerate(cos_sim_score[0]):
    corpus_id = ir["corpus_id"]
    print('corpus id: ', corpus_id)
    print(f'Document {no + 1}: Cosine Similarity is {ir["score"]:.3f}:\n\n{docs[corpus_id]}')
    print()

Question:  Какое содержание селена в морской воде?

corpus id:  561
Document 1: Cosine Similarity is 0.831:

Содержание селена в земной коре — около 500 мг/т. Основные черты геохимии селена в земной коре определяются близостью его ионного радиуса к ионному радиусу серы. Селен образует 37 минералов, среди которых в первую очередь должны быть отмечены ашавалит FeSe, клаусталит PbSe, тиманнит HgSe, гуанахуатит Bi2(Se, S)3, хастит CoSe2, платинит PbBi2(S, Se)3, ассоциирующие с различными сульфидами, а иногда также с касситеритом. Изредка встречается самородный селен. Главное промышленное значение на селен имеют сульфидные месторождения. Содержание селена в сульфидах колеблется от 7 до 110 г/т. Концентрация селена в морской воде 4·10−4 мг/л[15]. На территории Кавказских Минеральных Вод встречаются источники с содержанием Se от 50 мкг/дм3.

corpus id:  552
Document 2: Cosine Similarity is 0.776:

Подземные воды распространены на всей территории Восточно-Европейской равнины, образуя Восточно-

In [ ]:
data = test_shuffled_data(from_file=True)
n_test_samples = 1000
queries, docs, labels = zip(*data[:n_test_samples])

queries, docs, labels = shrink_repeated_samples(
    queries=queries,
    docs=docs,
    labels=labels,
)
print(len(docs))

891


# Получение предсказаний модели и меток

In [ ]:
doc_embs = finetuned_bi_encoder.encode(
    docs,
    convert_to_tensor=True,
    show_progress_bar=True,
)

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

In [ ]:
query_embs = finetuned_bi_encoder.encode(
    queries,
    convert_to_tensor=True,
    show_progress_bar=True,
)

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

In [ ]:
cos_scores = util.semantic_search(query_embs, doc_embs, top_k=100)

In [ ]:
print(len(cos_scores))

891


In [ ]:
print(cos_scores[0][:5])

[{'corpus_id': 0, 'score': 0.8310010433197021}, {'corpus_id': 671, 'score': 0.7761925458908081}, {'corpus_id': 183, 'score': 0.7383377552032471}, {'corpus_id': 392, 'score': 0.7231197357177734}, {'corpus_id': 244, 'score': 0.7215971350669861}]


In [ ]:
top_k = 5

In [ ]:
print('Question: ', queries[0])
print()
for no, ir in enumerate(cos_scores[0][:top_k]):
    corpus_id = ir["corpus_id"]
    print('corpus id: ', corpus_id)
    print(f'Document {no + 1}: Cosine Similarity is {ir["score"]:.3f}:\n\n{docs[corpus_id]}')
    print()

Question:  Какое содержание селена в морской воде?

corpus id:  0
Document 1: Cosine Similarity is 0.831:

Содержание селена в земной коре — около 500 мг/т. Основные черты геохимии селена в земной коре определяются близостью его ионного радиуса к ионному радиусу серы. Селен образует 37 минералов, среди которых в первую очередь должны быть отмечены ашавалит FeSe, клаусталит PbSe, тиманнит HgSe, гуанахуатит Bi2(Se, S)3, хастит CoSe2, платинит PbBi2(S, Se)3, ассоциирующие с различными сульфидами, а иногда также с касситеритом. Изредка встречается самородный селен. Главное промышленное значение на селен имеют сульфидные месторождения. Содержание селена в сульфидах колеблется от 7 до 110 г/т. Концентрация селена в морской воде 4·10−4 мг/л[15]. На территории Кавказских Минеральных Вод встречаются источники с содержанием Se от 50 мкг/дм3.

corpus id:  671
Document 2: Cosine Similarity is 0.776:

Подземные воды распространены на всей территории Восточно-Европейской равнины, образуя Восточно-Ев

In [ ]:
print(docs[582])

К хлоридно-гидрокарбонатно-натриевому типу относится закарпатская вода Драговская с минерализацией 9,6 г/л и краснодарская Горячий ключ с общим содержанием солей в литре 4,5 г солей, но в них хлориды в виде поваренной соли (соответственно 59 и 67 %) превалируют над гидрокарбонатами, которые представлены содой (38 и 32 %). Обе воды безсульфатные. Преобладанием хлоридов над гидрокарбонатами отличается и вода того же типа Челкарская с минерализацией 2,2 г/л. Гидрокарбонаты в виде соды составляют 32, а хлоридов (поваренная соль) — 48 %. Кроме того, в Челкарской есть сульфаты в виде глауберовой соли (20 %).


In [ ]:
n = len(queries)
print(n)

891


In [ ]:
def build_target_mask_for_i(i, n):
    target_mask = [False] * n
    target_mask[i] = True

    return target_mask

In [ ]:
def build_pred_mask_for_i(
        cos_scores,
        i,
        n,
):
    pred_mask = [0.0] * n
    qi_scores = cos_scores[i]

    for docid_score in qi_scores:
        doc_id = docid_score['corpus_id']
        score = docid_score['score']
        pred_mask[int(doc_id)] = score

    return pred_mask

# Расчёт метрик: Recall@5, MRR, MAP, NDCG@10

In [ ]:
from statistics import mean
from torchmetrics.functional.retrieval import retrieval_reciprocal_rank, retrieval_average_precision, \
    retrieval_normalized_dcg, retrieval_recall

r5s = []
mrrs = []
rmaps = []
ndcgs = []
for i in range(n):
    pred = build_pred_mask_for_i(cos_scores, i=i, n=n)
    target = build_target_mask_for_i(i=i, n=n)

    pred = torch.tensor(pred)
    target = torch.tensor(target)
    
    r5 = retrieval_recall(preds=pred, target=target, top_k=5).item()
    mrr = retrieval_reciprocal_rank(preds=pred, target=target, top_k=5).item()
    rmap = retrieval_average_precision(preds=pred, target=target, top_k=5).item()
    ndcg = retrieval_normalized_dcg(preds=pred, target=target, top_k=10).item()
    
    r5s.append(r5)
    mrrs.append(mrr)
    rmaps.append(rmap)
    ndcgs.append(ndcg)

print('mean recall5@891: ', mean(r5s))
print('mean mrr: ', mean(mrrs))
print('mean rmAP: ', mean(rmaps))
print('mean ndcg10@891: ', mean(ndcgs))

mean recall5@891:  0.4051627384960718
mean mrr:  0.34577254051244355
mean rmAP:  0.34577254051244355
mean ndcg10@891:  0.3706617497166934
